In [6]:
from tensorflow.keras import layers
from tensorflow.keras.layers import TimeDistributed, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

from keras.utils import to_categorical
from keras.callbacks import ModelCheckpoint
from keras.models import Sequential
from keras.layers import Conv2D,MaxPool2D,Flatten,LSTM, Dropout, Dense, TimeDistributed
import kapre
from kapre.composed import get_melspectrogram_layer
import tensorflow as tf
import os
from python_speech_features import mfcc
import pandas as pd
import numpy as np

from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
from scipy.io import wavfile

import pickle
from cfg import Config
import librosa

In [2]:
# def Conv1D(N_CLASSES=10, SR=16000, DT=1.0):
#     input_shape = (int(SR*DT), 1)
#     i = get_melspectrogram_layer(input_shape=input_shape,
#                                  n_mels=128,
#                                  pad_end=True,
#                                  n_fft=512,
#                                  win_length=400,
#                                  hop_length=160,
#                                  sample_rate=SR,
#                                  return_decibel=True,
#                                  input_data_format='channels_last',
#                                  output_data_format='channels_last')
#     x = LayerNormalization(axis=2, name='batch_norm')(i.output)
#     x = TimeDistributed(layers.Conv1D(8, kernel_size=(4), activation='tanh'), name='td_conv_1d_tanh')(x)
#     x = layers.MaxPooling2D(pool_size=(2,2), name='max_pool_2d_1')(x)
#     x = TimeDistributed(layers.Conv1D(16, kernel_size=(4), activation='relu'), name='td_conv_1d_relu_1')(x)
#     x = layers.MaxPooling2D(pool_size=(2,2), name='max_pool_2d_2')(x)
#     x = TimeDistributed(layers.Conv1D(32, kernel_size=(4), activation='relu'), name='td_conv_1d_relu_2')(x)
#     x = layers.MaxPooling2D(pool_size=(2,2), name='max_pool_2d_3')(x)
#     x = TimeDistributed(layers.Conv1D(64, kernel_size=(4), activation='relu'), name='td_conv_1d_relu_3')(x)
#     x = layers.MaxPooling2D(pool_size=(2,2), name='max_pool_2d_4')(x)
#     x = TimeDistributed(layers.Conv1D(128, kernel_size=(4), activation='relu'), name='td_conv_1d_relu_4')(x)
#     x = layers.GlobalMaxPooling2D(name='global_max_pooling_2d')(x)
#     x = layers.Dropout(rate=0.1, name='dropout')(x)
#     x = layers.Dense(64, activation='relu', activity_regularizer=l2(0.001), name='dense')(x)
#     o = layers.Dense(N_CLASSES, activation='softmax', name='softmax')(x)
#     model = Model(inputs=i.input, outputs=o, name='1d_convolution')
#     model.compile(optimizer='adam',
#                   loss='categorical_crossentropy',
#                   metrics=['accuracy'])
#     return model


# def Conv2D(N_CLASSES=10, SR=16000, DT=1.0):
#     input_shape = (int(SR*DT), 1)
#     i = get_melspectrogram_layer(input_shape=input_shape,
#                                  n_mels=128,
#                                  pad_end=True,
#                                  n_fft=512,
#                                  win_length=400,
#                                  hop_length=160,
#                                  sample_rate=SR,
#                                  return_decibel=True,
#                                  input_data_format='channels_last',
#                                  output_data_format='channels_last')
#     x = LayerNormalization(axis=2, name='batch_norm')(i.output)
#     x = layers.Conv2D(8, kernel_size=(7,7), activation='tanh', padding='same', name='conv2d_tanh')(x)
#     x = layers.MaxPooling2D(pool_size=(2,2), padding='same', name='max_pool_2d_1')(x)
#     x = layers.Conv2D(16, kernel_size=(5,5), activation='relu', padding='same', name='conv2d_relu_1')(x)
#     x = layers.MaxPooling2D(pool_size=(2,2), padding='same', name='max_pool_2d_2')(x)
#     x = layers.Conv2D(16, kernel_size=(3,3), activation='relu', padding='same', name='conv2d_relu_2')(x)
#     x = layers.MaxPooling2D(pool_size=(2,2), padding='same', name='max_pool_2d_3')(x)
#     x = layers.Conv2D(32, kernel_size=(3,3), activation='relu', padding='same', name='conv2d_relu_3')(x)
#     x = layers.MaxPooling2D(pool_size=(2,2), padding='same', name='max_pool_2d_4')(x)
#     x = layers.Conv2D(32, kernel_size=(3,3), activation='relu', padding='same', name='conv2d_relu_4')(x)
#     x = layers.Flatten(name='flatten')(x)
#     x = layers.Dropout(rate=0.2, name='dropout')(x)
#     x = layers.Dense(64, activation='relu', activity_regularizer=l2(0.001), name='dense')(x)
#     o = layers.Dense(N_CLASSES, activation='softmax', name='softmax')(x)
#     model = Model(inputs=i.input, outputs=o, name='2d_convolution')
#     model.compile(optimizer='adam',
#                   loss='categorical_crossentropy',
#                   metrics=['accuracy'])
#     return model


# def LSTM(N_CLASSES=10, SR=16000, DT=1.0):
#     input_shape = (int(SR*DT), 1)
#     i = get_melspectrogram_layer(input_shape=input_shape,
#                                      n_mels=128,
#                                      pad_end=True,
#                                      n_fft=512,
#                                      win_length=400,
#                                      hop_length=160,
#                                      sample_rate=SR,
#                                      return_decibel=True,
#                                      input_data_format='channels_last',
#                                      output_data_format='channels_last',
#                                      name='2d_convolution')
#     x = LayerNormalization(axis=2, name='batch_norm')(i.output)
#     x = TimeDistributed(layers.Reshape((-1,)), name='reshape')(x)
#     s = TimeDistributed(layers.Dense(64, activation='tanh'),
#                         name='td_dense_tanh')(x)
#     x = layers.Bidirectional(layers.LSTM(32, return_sequences=True),
#                              name='bidirectional_lstm')(s)
#     x = layers.concatenate([s, x], axis=2, name='skip_connection')
#     x = layers.Dense(64, activation='relu', name='dense_1_relu')(x)
#     x = layers.MaxPooling1D(name='max_pool_1d')(x)
#     x = layers.Dense(32, activation='relu', name='dense_2_relu')(x)
#     x = layers.Flatten(name='flatten')(x)
#     x = layers.Dropout(rate=0.2, name='dropout')(x)
#     x = layers.Dense(32, activation='relu',
#                          activity_regularizer=l2(0.001),
#                          name='dense_3_relu')(x)
#     o = layers.Dense(N_CLASSES, activation='softmax', name='softmax')(x)
#     model = Model(inputs=i.input, outputs=o, name='long_short_term_memory')
#     model.compile(optimizer='adam',
#                   loss='categorical_crossentropy',
#                   metrics=['accuracy'])

#     return model



In [3]:
data_dir = "../../../RAVDESS"
save_dir = "../../dataset/"
clean_dir = save_dir+"clean/"

def get_conv_model():
    model = Sequential()
    model.add(Conv2D(16,(3,3),activation="relu",strides=(1,1),
                    padding="same",input_shape=input_shape))
    model.add(Conv2D(32,(3,3),activation="relu",strides=(1,1),
             padding="same"))
    model.add(Conv2D(64,(3,3),activation="relu",strides=(1,1),
             padding="same"))
    model.add(Conv2D(128,(3,3),activation="relu",strides=(1,1),
             padding="same"))
    model.add(MaxPool2D((2,2)))
    model.add(Dropout(0.5))
    model.add(Flatten())
    model.add(Dense(128,activation="relu"))
    model.add(Dense(64,activation="relu"))
    model.add(Dense(8,activation="softmax"))
    print(model.summary())
    model.compile(loss="categorical_crossentropy",
                  optimizer="adam",
                 metrics=["acc"])
    return model

def get_recurrent_model():
    #shape of RNN is (n,time,feat)
    model = Sequential()
    model.add(LSTM(128,return_sequences=True,input_shape=input_shape))
    model.add(LSTM(128,return_sequences=True))
    model.add(Dropout(0.5))
    model.add(TimeDistributed(Dense(64,activation="relu")))
    model.add(TimeDistributed(Dense(32,activation="relu")))
    model.add(TimeDistributed(Dense(16,activation="relu")))
    model.add(TimeDistributed(Dense(8,activation="relu")))
    model.add(Flatten())
    model.add(Dense(8,activation="softmax"))
    print(model.summary())
    model.compile(loss="categorical_crossentropy",
                  optimizer="adam",
                 metrics=["acc"])
    return model

def check_data():
    if os.path.isfile(config.p_path):
        print(f"Loading existing data for {config.mode} model")
        with open(config.p_path,"rb") as handle:
            tmp = pickle.load(handle)
            return tmp
    else:
        return None
def build_rand_feat():
    tmp = check_data()
    if tmp:
        return tmp.data[0],tmp.data[1]
    
    X = []
    y = []
    _min, _max = float("inf"), float("-inf")
    
    for _ in tqdm(range(n_samples)):
        rand_class = np.random.choice(class_dist.index,p=prob_dist)
        file = np.random.choice(df[df["emotion"]==rand_class]["filename"])
        label = df[df["filename"]==file]["emotion"].iloc[0]
        rate,wav = wavfile.read(clean_dir+file)
        rand_index = np.random.randint(0,wav.shape[0]-config.step)
        
        sample = wav[rand_index:rand_index+config.step]
        X_sample = mfcc(sample,rate,numcep=config.nfeat,nfilt=config.nfilt,nfft=config.nfft)
        
        _min = min(np.amin(X_sample),_min)
        _max = max(np.amax(X_sample),_max)
        
#         X.append(X_sample if config.mode == "conv" else X_sample.T)
        X.append(X_sample)
        y.append(classes.index(label))
        
    config.min = _min
    config.max = _max
    X,y = np.array(X), np.array(y)
    X = (X - _min) / (_max - _min)
    
    if config.mode == "conv":
        X = X.reshape(X.shape[0],X.shape[1],X.shape[2],1)
    elif config.mode == "time":
        X = X.reshape(X.shape[0], X.shape[1],X.shape[2])
        
    y = to_categorical(y, num_classes=8)
    config.data = (X,y)
    
    with open(config.p_path,"wb") as handle:
        pickle.dump(config,handle,protocol=2)
    return X,y

In [4]:
df = pd.read_csv("audio_data.csv")
classes = list(df["emotion"].unique())
class_dist = df.groupby(["emotion"])["length"].mean()

prob_dist = class_dist/ class_dist.sum()
choices = np.random.choice(class_dist.index, p = prob_dist)

n_samples = 2 * int(df["length"].sum()/0.1)

In [98]:
class Config:
    def __init__(self,mode="conv",nfilt=26,nfeat=13,nfft=512,rate=16000):
        self.mode = mode
        self.nfilt = nfilt
        self.nfeat = nfeat
        self.nfft = nfft
        self.rate = rate
        self.step = int(rate/10)
        self.model_path = os.path.join("models",mode+".model")
        self.p_path = os.path.join("pickles",mode+".p")
        

config = Config()

if config.mode == "conv":
    X,y = build_rand_feat()
    y_flat = np.argmax(y,axis=1)
    input_shape = (X.shape[1],X.shape[2],1)
    model = get_conv_model()
elif config.mode == "time":
    X,y = build_rand_feat()
    y_flat = np.argmax(y,axis=1)
    input_shape = (X.shape[1],X.shape[2])
    model = get_recurrent_model()

# class_weight = compute_class_weight("balanced",np.unique(y_flat),y_flat)
checkpoint = ModelCheckpoint(config.model_path,monitor="val_acc",verbose=1,mode="max",save_best_only=True,save_weights_only=False,period=1)
model.fit(X,y,epochs=10,batch_size=32,
          shuffle=True,validation_split=0.1,callbacks=[checkpoint])

model.save(config.model_path)

100%|█████████████████████████████████████████████████████████████████████████| 106578/106578 [04:08<00:00, 429.49it/s]


Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_8 (Conv2D)           (None, 9, 13, 16)         160       
                                                                 
 conv2d_9 (Conv2D)           (None, 9, 13, 32)         4640      
                                                                 
 conv2d_10 (Conv2D)          (None, 9, 13, 64)         18496     
                                                                 
 conv2d_11 (Conv2D)          (None, 9, 13, 128)        73856     
                                                                 
 max_pooling2d_2 (MaxPooling  (None, 4, 6, 128)        0         
 2D)                                                             
                                                                 
 dropout_2 (Dropout)         (None, 4, 6, 128)         0         
                                                      

Epoch 1/10
2994/2998 [============================>.] - ETA: 0s - loss: 1.9230 - acc: 0.2344
Epoch 1: val_acc improved from -inf to 0.29114, saving model to models\conv.model


INFO:tensorflow:Assets written to: models\conv.model\assets


INFO:tensorflow:Assets written to: models\conv.model\assets


...

INFO:tensorflow:Assets written to: models\conv.model\assets


INFO:tensorflow:Assets written to: models\conv.model\assets



2998/2998 [==============================] - 24s 8ms/step - loss: 1.4475 - acc: 0.4536 - val_loss: 1.4330 - val_acc: 0.4636


INFO:tensorflow:Assets written to: models\conv.model\assets


INFO:tensorflow:Assets written to: models\conv.model\assets


In [9]:
# X,y = build_rand_feat()
# y_flat = np.argmax(y,axis=1)
# input_shape = (X.shape[1],X.shape[2],1)
model = get_conv_model()
checkpoint = ModelCheckpoint("conv.model",monitor="val_acc",verbose=1,mode="max",save_best_only=True,save_weights_only=False,save_freq=1)
model.fit(X,y,epochs=10,batch_size=32,
          shuffle=True,validation_split=0.1,callbacks=[checkpoint])

model.save("conv.model")

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 9, 13, 16)         160       
                                                                 
 conv2d_1 (Conv2D)           (None, 9, 13, 32)         4640      
                                                                 
 conv2d_2 (Conv2D)           (None, 9, 13, 64)         18496     
                                                                 
 conv2d_3 (Conv2D)           (None, 9, 13, 128)        73856     
                                                                 
 max_pooling2d (MaxPooling2D  (None, 4, 6, 128)        0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 4, 6, 128)         0         
                                                      

INFO:tensorflow:Assets written to: conv.model\assets


INFO:tensorflow:Assets written to: conv.model\assets


2998/2998 [==============================] - 39s 8ms/step - loss: 1.9289 - acc: 0.2306 - val_loss: 1.8002 - val_acc: 0.2942
Epoch 2/10
2997/2998 [============================>.] - ETA: 0s - loss: 1.7867 - acc: 0.3024
Epoch 2: val_acc improved from 0.29424 to 0.33477, saving model to conv.model


INFO:tensorflow:Assets written to: conv.model\assets


...

INFO:tensorflow:Assets written to: conv.model\assets


INFO:tensorflow:Assets written to: conv.model\assets



2998/2998 [==============================] - 21s 7ms/step - loss: 1.4317 - acc: 0.4609 - val_loss: 1.4293 - val_acc: 0.4750


INFO:tensorflow:Assets written to: conv.model\assets


INFO:tensorflow:Assets written to: conv.model\assets


In [80]:
prediction = new_model.predict(X_sample.reshape(1,X_sample.shape[0],X_sample.shape[1],1))[0]
predict = int(np.where(np.isclose(prediction,1))[0])
classes[predict]

1/1 [==============================] - 0s 22ms/step


'angry'

In [91]:
for x in prediction:
    print(x)

1.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


In [82]:
classes

['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

In [10]:
class Config:
    def __init__(self,mode="conv",nfilt=26,nfeat=13,nfft=512,rate=16000):
        self.mode = mode
        self.nfilt = nfilt
        self.nfeat = nfeat
        self.nfft = nfft
        self.rate = rate
        self.step = int(rate/10)
        self.model_path = os.path.join("models",mode+".model")
        self.p_path = os.path.join("pickles",mode+".p")
        
config = Config()      
new_model = tf.keras.models.load_model('conv.model')

# Check its architecture
new_model.summary()

file = os.listdir(clean_dir)[0]
rate,wav = wavfile.read(clean_dir+file)
# rand_index = np.random.randint(0,wav.shape[0]-config.step)
        
# sample = wav[rand_index:rand_index+config.step]
X_sample = mfcc(wav,rate,numcep=config.nfeat,nfilt=config.nfilt,nfft=config.nfft)
prediction = new_model.predict(X_sample.reshape(1,X_sample.shape[0],X_sample.shape[1],1))[0]
predict = int(np.where(np.isclose(prediction,1))[0])
classes[predict],file

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 9, 13, 16)         160       
                                                                 
 conv2d_1 (Conv2D)           (None, 9, 13, 32)         4640      
                                                                 
 conv2d_2 (Conv2D)           (None, 9, 13, 64)         18496     
                                                                 
 conv2d_3 (Conv2D)           (None, 9, 13, 128)        73856     
                                                                 
 max_pooling2d (MaxPooling2D  (None, 4, 6, 128)        0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 4, 6, 128)         0         
                                                      

ValueError: in user code:

    File "C:\Users\User\anaconda3\lib\site-packages\keras\engine\training.py", line 2041, in predict_function  *
        return step_function(self, iterator)
    File "C:\Users\User\anaconda3\lib\site-packages\keras\engine\training.py", line 2027, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "C:\Users\User\anaconda3\lib\site-packages\keras\engine\training.py", line 2015, in run_step  **
        outputs = model.predict_step(data)
    File "C:\Users\User\anaconda3\lib\site-packages\keras\engine\training.py", line 1983, in predict_step
        return self(x, training=False)
    File "C:\Users\User\anaconda3\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "C:\Users\User\anaconda3\lib\site-packages\keras\engine\input_spec.py", line 295, in assert_input_compatibility
        raise ValueError(

    ValueError: Input 0 o

In [27]:
classes = list(df["emotion"].unique())
fn2class = dict(zip(df["filename"],df["emotion"]))
p_path = os.path.join("pickles","conv.p")

with open(p_path,"rb") as handle:
    config = pickle.load(handle)
model = tf.keras.models.load_model('conv.model')

# Check its architecture
model.summary()

file = os.listdir(clean_dir)[0]
rate,wav = wavfile.read(clean_dir+file)

# y_true = []
# y_pred = []
y_prob = []
        
for i in range(0,wav.shape[0]-config.step,config.step):
    sample = wav[i:i+config.step]
    x = mfcc(sample,rate,numcep=config.nfeat,
                     nfilt=config.nfilt,nfft=config.nfft)
    x = (x-config.min)/(config.max - config.min)
            
    if config.mode == "conv":
        x = x.reshape(1,x.shape[0],x.shape[1],1)
    elif config.mode == "time":
        x = np.expand_dims(x_axis=0)
    c = fn2class[file]
    y_hat = model.predict(x)
    y_prob.append(y_hat)
#     y_pred.append(np.argmax(y_hat))
#     y_true.append(c)

fn_prob = np.mean(y_prob,axis=0).flatten()
pred = [classes[np.argmax(y)] for y in y_prob]
class_dict = {'neutral':[], 'calm':[], 'happy':[], 'sad':[], 'angry':[], 'fearful':[], 'disgust':[], 'surprised':[]}
for c,p in zip(classes,fn_prob):
    class_dict[c].append(p)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 9, 13, 16)         160       
                                                                 
 conv2d_1 (Conv2D)           (None, 9, 13, 32)         4640      
                                                                 
 conv2d_2 (Conv2D)           (None, 9, 13, 64)         18496     
                                                                 
 conv2d_3 (Conv2D)           (None, 9, 13, 128)        73856     
                                                                 
 max_pooling2d (MaxPooling2D  (None, 4, 6, 128)        0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 4, 6, 128)         0         
                                                      

In [16]:
fn_prob

array([0.4642533 , 0.23780131, 0.033109  , 0.14307998, 0.02465682,
       0.03275897, 0.04194259, 0.02239802], dtype=float32)

In [28]:
pred

['neutral',
 'neutral',
 'neutral',
 'neutral',
 'neutral',
 'calm',
 'calm',
 'neutral',
 'calm',
 'neutral',
 'calm',
 'calm',
 'neutral',
 'neutral',
 'neutral']

In [26]:
y_pred

[0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0]

In [23]:
classes

['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

In [24]:
class_dict = {'neutral':[], 'calm':[], 'happy':[], 'sad':[], 'angry':[], 'fearful':[], 'disgust':[], 'surprised':[]}
for c,p in zip(classes,fn_prob):
    class_dict[c].append(p)

In [25]:
class_dict

{'neutral': [0.4642533],
 'calm': [0.23780131],
 'happy': [0.033109],
 'sad': [0.14307998],
 'angry': [0.024656825],
 'fearful': [0.032758974],
 'disgust': [0.04194259],
 'surprised': [0.022398015]}

In [19]:
y_hat

array([[0.8640737 , 0.05819903, 0.00979021, 0.01933831, 0.00135374,
        0.02200115, 0.00979626, 0.01544761]], dtype=float32)

In [20]:
np.argmax(y_hat)

0